# Janus colloids — learning general pair interactions from experimental tracks

Self-propelled Janus colloids interact through electrostatic and
hydrodynamic forces that cannot be computed from first principles.  Hem,
Poncet, Ronceray, Nishiguchi and Démery (*Soft Matter* 2025,
`doi:10.1039/D5SM00655D <https://doi.org/10.1039/D5SM00655D>`_) learned
them from tracking data instead, with a **generic** basis: a Fourier
series in the two relative orientations times smooth radial functions,
for the radial velocity, the transverse velocity and the torque that one
colloid induces on another.

This page redoes that analysis with SFI on the experimental tracks the
authors distribute with their code — the first 100 frames of their
flocking dataset (set A, Iwasawa, Nishiguchi & Sano, *Phys. Rev. Research*
2021), about 1 700 colloids at 15 frames per second — then simulates the
learned model to see whether it flocks.

**Tags**

real-data · overdamped · multi-particle · linear · custom-basis · 2D · non-equilibrium

In [ ]:
from __future__ import annotations

import os
os.environ["JAX_PLATFORMS"] = "cpu"

import warnings
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from SFI.utils.plotting import SFI_COLORS, stamp_output
from SFI import TrajectoryCollection
from SFI.utils.plotting import animate_particles, animate_particles_comparison, dark_fig

plt.rcParams["savefig.facecolor"] = "auto"   # movies on their own dark background
warnings.formatwarning = lambda msg, cat, *args, **kw: f"{cat.__name__}: {msg}\n"
warnings.filterwarnings("ignore", message=r".*Cartesian product over feature axes.*")

cache_dir = Path(__file__).resolve().parent / "_cache" if "__file__" in dir() else Path("_cache")
SIGMA = 3.17                                        # particle diameter (µm)
TH = np.linspace(0, 2 * np.pi, 97)
T1, T2 = np.meshgrid(TH, TH, indexing="ij")
TITLES = [r"$v_r/U$", r"$v_\theta/U$", r"$\sigma\omega/U$"]

def _angle_axes(ax):
    ticks = [0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi]
    labels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]
    ax.set_xticks(ticks); ax.set_xticklabels(labels)
    ax.set_yticks(ticks); ax.set_yticklabels(labels)
    ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")

def _particle_axes(axes, fov):
    for ax in np.atleast_1d(axes):
        ax.set_xlim(0, fov[0]); ax.set_ylim(0, fov[1]); ax.set_aspect("equal")
        ax.set_xticks([]); ax.set_yticks([])
        for col in ax.collections:
            col.set_edgecolor("none")
stamp_output()

Helper code from the SFI gallery (``examples/_gallery_utils/``), inlined so that this notebook runs on its own.

In [ ]:
# --- from examples/_gallery_utils/janus_colloids.py ---
import hashlib

import pickle

import urllib.request

from pathlib import Path

import numpy as np

from scipy.spatial import cKDTree

try:
    from numpy._core.multiarray import _reconstruct
except ImportError:  # numpy < 2
    from numpy.core.multiarray import _reconstruct

DATA_URL = ("https://raw.githubusercontent.com/jhem272/sfiabp_py/"
            "7bf78a5298c87cbd40a211d61e26e65525a1b6d2/tutorial/data/"
            "20180927ordered_df_pv_m_100f.pkl")

DATA_SHA256 = "7d7ef2460ceee5f83a4dc42df4f4ba60849fe6d32c6ed18089afb13c319ec542"

def _create_array(f, args, state, npdict=None):
    """Stand-in for ``dill._dill._create_array`` (rebuilds one numpy array)."""
    a = f(*args)
    a.__setstate__(state)
    if npdict:
        a.__dict__.update(npdict)
    return a

class _ArraysOnlyUnpickler(pickle.Unpickler):
    """Unpickler that only rebuilds numpy arrays: no other code can run."""

    def find_class(self, module, name):
        if (module, name) == ("dill._dill", "_create_array"):
            return _create_array
        if module in ("numpy.core.multiarray", "numpy._core.multiarray") and name == "_reconstruct":
            return _reconstruct
        if (module, name) in (("numpy", "ndarray"), ("numpy", "dtype")):
            return getattr(np, name)
        raise pickle.UnpicklingError(f"refusing to load {module}.{name}")

def load_set_a(cache_dir: Path):
    """Download (once) and read the set-A tracks.

    Returns
    -------
    frames : list of arrays ``(N_t, 4)``
        Per frame: ``x, y`` (µm), heading ``φ`` (rad, in ``[0, 2π)``) and
        the integer track label of each detected particle.
    dt : float
        Frame interval (s).
    fov : ndarray ``(2,)``
        Width and height of the field of view (µm).
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / "janus_set_a_100f.pkl"
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL, path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != DATA_SHA256:
        path.unlink()
        raise RuntimeError("checksum mismatch for the downloaded tracks; removed the file")
    with open(path, "rb") as fh:
        d = _ArraysOnlyUnpickler(fh).load()
    fov = np.array([d["xframelim"][1], d["yframelim"][1]], dtype=float)
    return [np.asarray(f, dtype=float) for f in d["X_Raw"]], float(d["dtframe"]), fov

def tracks_to_arrays(frames):
    """Dense ``(T, N, 3)`` states, one column per track label, plus the mask.

    Absent particles are NaN with ``mask = False``.  Headings are unwrapped
    along each track so that heading increments are continuous.
    """
    labels = np.unique(np.concatenate([f[:, 3] for f in frames]).astype(int))
    slot = {lab: i for i, lab in enumerate(labels)}
    T, N = len(frames), len(labels)
    X = np.full((T, N, 3), np.nan)
    mask = np.zeros((T, N), bool)
    for t, f in enumerate(frames):
        s = np.array([slot[int(lab)] for lab in f[:, 3]])
        X[t, s] = f[:, :3]
        mask[t, s] = True
    for n in range(N):
        X[mask[:, n], n, 2] = np.unwrap(X[mask[:, n], n, 2])
    return X, mask

def per_frame_neighbours(X, mask, radius):
    """Per-frame CSR neighbour lists over the present particles, as extras."""
    from SFI.trajectory import TimeSeriesExtra
    from SFI.utils.neighbors import build_neighbor_csr, pad_neighbor_csr

    T, N = mask.shape
    indptrs, indices = [], []
    for t in range(T):
        present = np.flatnonzero(mask[t])
        ip, ix = build_neighbor_csr(X[t, present, :2], radius)
        counts = np.zeros(N, int)
        counts[present] = np.diff(ip)
        indptrs.append(np.concatenate([[0], np.cumsum(counts)]).astype(np.int32))
        indices.append(present[ix].astype(np.int32))
    nnz = max(len(ix) for ix in indices)
    indices = [pad_neighbor_csr(ip, ix, nnz)[1] for ip, ix in zip(indptrs, indices)]
    return {"indptr": TimeSeriesExtra(np.stack(indptrs)), "indices": TimeSeriesExtra(np.stack(indices))}

def interior_rows(X, mask, fov, margin, lag=2):
    """Fit mask: particles at least ``margin`` from the edge ``lag`` frames earlier."""
    x, y = X[..., 0], X[..., 1]
    with np.errstate(invalid="ignore"):
        inner = mask & (x > margin) & (x < fov[0] - margin) & (y > margin) & (y < fov[1] - margin)
    rows = np.zeros_like(mask)
    rows[lag:] = inner[:-lag]
    return rows & mask

def _tiles(frame, fov, side):
    """Tile index of each particle (square tiles of the given side) and tile count."""
    nx, ny = int(fov[0] // side), int(fov[1] // side)
    xy = np.mod(frame[:, :2], fov)
    ix, iy = (xy[:, 0] // side).astype(int), (xy[:, 1] // side).astype(int)
    ok = (ix < nx) & (iy < ny)
    return ix * ny + iy, ok, nx * ny

def polar_order_vs_area(frames, fov, sides):
    r"""Polar order :math:`C(S) = \langle|\langle e^{i\varphi}\rangle_{\rm tile}|\rangle` for tiles of area side²."""
    out = []
    for side in sides:
        vals = []
        for f in frames:
            tile, ok, n = _tiles(f, fov, side)
            z = np.exp(1j * f[ok, 2])
            s = np.bincount(tile[ok], z.real, n) + 1j * np.bincount(tile[ok], z.imag, n)
            cnt = np.bincount(tile[ok], None, n)
            vals.append(np.abs(s[cnt > 0]) / cnt[cnt > 0])
        out.append(np.mean(np.concatenate(vals)))
    return np.array(out)

def polarity_autocorrelation(phi, lags):
    r""":math:`f_p(\tau) = \langle \cos[\varphi(t+\tau) - \varphi(t)]\rangle` from headings ``(T, N)`` (NaN = absent)."""
    return np.array([np.nanmean(np.cos(phi[k:] - phi[:len(phi) - k])) for k in lags])

def number_fluctuations(frames, fov, sides):
    """Mean and standard deviation of the particle number in square tiles."""
    mean, std = [], []
    for side in sides:
        counts = []
        for f in frames:
            tile, ok, n = _tiles(f, fov, side)
            counts.append(np.bincount(tile[ok], None, n))
        counts = np.array(counts)
        mean.append(counts.mean())
        std.append(counts.std(axis=0).mean())
    return np.array(mean), np.array(std)

def body_frame_density(frames, fov, *, half_width=25.0, bins=50, periodic=False):
    r"""Neighbour density around a particle, in its own frame (:math:`x_\parallel` along its heading).

    Normalised by the mean density; with open boundaries only particles
    farther than the window from the edges are used as centres.
    """
    reach = half_width * np.sqrt(2)
    H = np.zeros((bins, bins))
    n_centres, density = 0, 0.0
    for f in frames:
        xy = np.mod(f[:, :2], fov) if periodic else f[:, :2]
        pairs = cKDTree(xy, boxsize=fov if periodic else None).query_pairs(reach, output_type="ndarray")
        pairs = np.concatenate([pairs, pairs[:, ::-1]])
        if periodic:
            n_centres += len(xy)
        else:
            inner = np.all((xy > reach) & (xy < fov - reach), axis=1)
            pairs = pairs[inner[pairs[:, 0]]]
            n_centres += inner.sum()
        d = xy[pairs[:, 1]] - xy[pairs[:, 0]]
        if periodic:
            d -= fov * np.round(d / fov)
        c, s = np.cos(f[pairs[:, 0], 2]), np.sin(f[pairs[:, 0], 2])
        H += np.histogram2d(c * d[:, 0] + s * d[:, 1], -s * d[:, 0] + c * d[:, 1], bins=bins,
                            range=[[-half_width, half_width]] * 2)[0]
        density += len(xy) / np.prod(fov)
    return H / (n_centres * density / len(frames) * (2 * half_width / bins) ** 2)

def relative_orientations(frames, r_lo, r_hi):
    r"""Angles :math:`(\theta_1, \theta_2)` of all ordered pairs with :math:`r_{lo} \le r < r_{hi}`.

    :math:`\theta_k = \varphi_k - \arg(\mathbf r_1 - \mathbf r_2)`, the
    convention of the paper (particle 2 faces particle 1 at :math:`\theta_2 = 0`).
    """
    t1, t2 = [], []
    for f in frames:
        pairs = cKDTree(f[:, :2]).query_pairs(r_hi, output_type="ndarray")
        pairs = np.concatenate([pairs, pairs[:, ::-1]])
        d = f[pairs[:, 0], :2] - f[pairs[:, 1], :2]
        r = np.hypot(d[:, 0], d[:, 1])
        keep = r >= r_lo
        phi_r = np.arctan2(d[keep, 1], d[keep, 0])
        t1.append(f[pairs[keep, 0], 2] - phi_r)
        t2.append(f[pairs[keep, 1], 2] - phi_r)
    return np.mod(np.concatenate(t1), 2 * np.pi), np.mod(np.concatenate(t2), 2 * np.pi)

def tracks_collection(frames, dt, fov, *, cutoff, margin, lag=2):
    """Experimental tracks as a :class:`~SFI.TrajectoryCollection`, ready for pair inference.

    One column per track label (absent particles masked), headings unwrapped,
    per-frame neighbour lists of radius ``cutoff`` + a Verlet skin measured on
    the tracks (twice the largest displacement over ``lag`` frames), and a
    fit mask (``dynamic_mask``) keeping particles farther than ``margin`` from
    the edges ``lag`` frames before the row; the others stay visible as
    neighbours.
    """
    from SFI import TrajectoryCollection, TrajectoryDataset

    X, mask = tracks_to_arrays(frames)
    skin = 2.0 * np.nanmax(np.hypot(*np.moveaxis(X[lag:, :, :2] - X[:-lag, :, :2], -1, 0)))
    ds = TrajectoryDataset.from_arrays(X=X, dt=dt, mask=mask, dynamic_mask=interior_rows(X, mask, fov, margin, lag),
                                       extras_global=per_frame_neighbours(X, mask, cutoff + skin))
    return TrajectoryCollection.from_dataset(ds)

def tapered_polyexp_kernels(*, cutoff, onset, degrees=range(0, 16, 2), r0=1.0, taper=4.0):
    r"""Radial functions :math:`g_l(r) = (r/r_0)^l e^{-r/r_0}/l!` of the paper's code, tapered.

    The window rises smoothly over ``onset`` (a pair of distances) and falls
    to zero, with its slope, over the last ``taper`` before ``cutoff``.
    Returns ``(callable, label)`` pairs for :func:`~SFI.bases.pairs.scalar_pair_basis`.
    """
    import jax.numpy as jnp
    from math import lgamma

    def step(u):
        u = jnp.clip(u, 0.0, 1.0)
        return u * u * (3.0 - 2.0 * u)

    lo, hi = onset

    def g(l):
        c = lgamma(l + 1)
        return lambda r: (jnp.exp(l * jnp.log(r / r0) - r / r0 - c)
                          * step((r - lo) / (hi - lo)) * step((cutoff - r) / taper))
    return [(g(l), f"g{l}") for l in degrees]

def angular_modes(Na):
    """``(k1, k2)`` with ``|k1| + |k2| <= Na``, one per pair ``±(k1, k2)``."""
    return [(n - abs(k), k) for n in range(Na + 1) for k in (range(-n + 1, n + 1) if n else [0])]

def _relative_frame(XK, extras, periodic):
    import jax.numpy as jnp
    from SFI.bases.pairs import pbc_displacement

    x1, x2 = XK[0], XK[1]
    d = -pbc_displacement(x2[:2], x1[:2], extras["box"][:2]) if periodic else x1[:2] - x2[:2]
    phi_r = jnp.arctan2(d[1], d[0])
    return phi_r, x1[2] - phi_r, x2[2] - phi_r, jnp.sqrt(jnp.sum(d ** 2))

def orientation_features(Na, *, box=None, sector=None):
    r"""Angular part of the paper's basis, as a rank-1 pair interactor on states ``(x, y, φ)``.

    Headings :math:`\theta_{1,2}` are measured from :math:`\mathbf r_1 - \mathbf r_2`;
    :math:`\cos(k_1\theta_1 + k_2\theta_2)` along :math:`\hat e_r`, the sines along
    :math:`\hat e_\theta` and on the heading (torque) — the parity-symmetric
    Fourier series of order ``Na``.  ``sector="translation"`` keeps the
    :math:`\hat e_r, \hat e_\theta` features, ``sector="rotation"`` the torque ones.
    """
    import jax.numpy as jnp
    from SFI.statefunc import make_interactor

    cos_m = angular_modes(Na)
    sin_m = [m for m in cos_m if m != (0, 0)]
    kc, ks = jnp.array(cos_m, float).reshape(-1, 2), jnp.array(sin_m, float).reshape(-1, 2)

    def local(XK, *, extras=None):
        phi_r, t1, t2, _ = _relative_frame(XK, extras, box == "extras")
        e_r = jnp.array([jnp.cos(phi_r), jnp.sin(phi_r), 0.0])
        e_t = jnp.array([-jnp.sin(phi_r), jnp.cos(phi_r), 0.0])
        e_w = jnp.array([0.0, 0.0, 1.0])
        c = jnp.cos(kc[:, 0] * t1 + kc[:, 1] * t2)
        s = jnp.sin(ks[:, 0] * t1 + ks[:, 1] * t2)
        parts = {"translation": [e_r[:, None] * c, e_t[:, None] * s], "rotation": [e_w[:, None] * s]}
        return jnp.concatenate(sum((parts[k] for k in keep), []), axis=1)

    keep = ("translation", "rotation") if sector is None else (sector,)
    names = {"translation": [f"vr cos({a}θ1{b:+d}θ2)" for a, b in cos_m] + [f"vθ sin({a}θ1{b:+d}θ2)" for a, b in sin_m],
             "rotation": [f"ω sin({a}θ1{b:+d}θ2)" for a, b in sin_m]}
    labels = sum((names[k] for k in keep), [])
    return make_interactor(local, dim=3, rank=1, K=2, n_features=len(labels), labels=labels,
                           extras_keys=("box",) if box == "extras" else ())

def interaction_reader(interactor):
    r"""Evaluate a pair interactor's features at given ``(r, θ1, θ2)``.

    Returns ``read(coef, cov, r, t1, t2) -> (value, stderr)``, each of shape
    ``(..., 3)`` for :math:`(v_r, v_\theta, \omega)`: the features are evaluated on a
    two-particle configuration (particle 2 at :math:`(-r, 0)`, so
    :math:`\hat e_r = \hat x`) and contracted with the coefficients.
    """
    import jax.numpy as jnp

    probe = interactor.dispatch_pairs(return_as="basis", exclude_self=True, owners="focal", reducer="sum")

    def read(coef, cov, r, t1, t2):
        r, t1, t2 = np.broadcast_arrays(np.asarray(r, float), t1, t2)
        X2 = np.zeros(r.shape + (2, 3))
        X2[..., 0, 2], X2[..., 1, 0], X2[..., 1, 2] = t1, -r, t2
        Phi = np.asarray(probe(jnp.asarray(X2.reshape(-1, 2, 3))))[:, 0]
        value = Phi @ np.ravel(coef)
        err = (np.zeros_like(value) if cov is None
               else np.sqrt(np.einsum("nca,ab,ncb->nc", Phi, np.asarray(cov), Phi)))
        return value.reshape(r.shape + (3,)), err.reshape(r.shape + (3,))

    return read

def fitted_pair_force(kernels, Na, coef):
    """The fitted series as a single periodic pair feature — fast to simulate."""
    import jax.numpy as jnp
    from SFI.statefunc import make_interactor

    fns = [fn for fn, _ in kernels]
    cos_m = angular_modes(Na)
    sin_m = [m for m in cos_m if m != (0, 0)]
    kc, ks = jnp.array(cos_m, float).reshape(-1, 2), jnp.array(sin_m, float).reshape(-1, 2)
    A = jnp.asarray(np.ravel(coef).reshape(len(fns), -1))
    nc, ns = len(cos_m), len(sin_m)

    def local(XK, *, extras=None):
        phi_r, t1, t2, r = _relative_frame(XK, extras, True)
        w = jnp.stack([g(r) for g in fns]) @ A
        s = jnp.sin(ks[:, 0] * t1 + ks[:, 1] * t2)
        vr = w[:nc] @ jnp.cos(kc[:, 0] * t1 + kc[:, 1] * t2)
        vt, om = w[nc:nc + ns] @ s, w[nc + ns:] @ s
        return jnp.array([vr * jnp.cos(phi_r) - vt * jnp.sin(phi_r),
                          vr * jnp.sin(phi_r) + vt * jnp.cos(phi_r), om])[:, None]

    return make_interactor(local, dim=3, rank=1, K=2, n_features=1, extras_keys=("box",),
                           labels=("fitted pair force",)).dispatch_pairs_from_extras(
        indptr_key="indptr", indices_key="indices", return_as="basis")

def simulate_fitted(inf, one_body, pair_block, kernels, Na, x0, duration, *, D, box, dt, key, cutoff,
                    sigma=3.17, eps_steric=800.0, n_sub=10, skin=6.0):
    """Run a fitted model from ``x0`` in a periodic box; one saved frame per ``dt``.

    The fitted pair series is summed into one feature (:func:`fitted_pair_force`)
    and a soft steric repulsion ``V = (ε/2)(1 - r/σ)²`` (as in the paper's
    simulations) keeps particles from overlapping.
    """
    import jax.numpy as jnp
    from SFI.bases.pairs import radial_pair_basis
    from SFI.langevin import OverdampedProcess
    from SFI.langevin.chunked import simulate_chunked
    from SFI.statefunc import Basis
    from SFI.utils.neighbors import make_neighbor_extras

    steric = radial_pair_basis([(lambda r: jnp.where(r < sigma, 1.0 - r / sigma, 0.0), "steric")], dim=3,
                               box="extras", spatial_dims=slice(0, 2), embed_dim=3, embed_axes=[0, 1]
                               ).dispatch_pairs_from_extras(indptr_key="indptr", indices_key="indices",
                                                            return_as="basis")
    c_one = [float(np.ravel(inf.coeff_block(b)[0])[0]) for b in one_body]
    F = Basis.stack([*one_body, fitted_pair_force(kernels, Na, inf.coeff_block(pair_block)[0]), steric])
    proc = OverdampedProcess(F=F, D=jnp.asarray(D), theta_F=jnp.array(c_one + [1.0, -eps_steric / sigma]),
                             extras_global={"box": jnp.asarray(box),
                                            **make_neighbor_extras(np.mod(x0[:, :2], box), cutoff + skin, box)})
    proc.initialize(jnp.asarray(x0))
    coll = simulate_chunked(proc, dt=dt / n_sub, Nsteps=int(round(duration / dt)) * n_sub, key=key,
                            cutoff=cutoff, skin=skin, box=np.asarray(box), rebuild_every=n_sub,
                            spatial_dims=slice(0, 2))
    return np.concatenate([np.asarray(x0)[None], np.asarray(coll.to_array(axis="time"))[n_sub - 1::n_sub]])

def residual_noise(inf, coll):
    r"""Measurement noise and diffusion from the residuals of a force fit.

    SFI's :math:`\hat\Lambda = -\langle \Delta X_t \Delta X_{t-1}\rangle` is
    biased by :math:`-\langle F_t F_{t-1}\rangle\Delta t^2` when the drift is
    persistent (self-propulsion).  On the residuals
    :math:`r_t = \Delta X_t - \hat F(X_t)\Delta t` the drift is gone:
    :math:`\Lambda = -\langle r_t r_{t-1}\rangle` and
    :math:`D = \langle r_t r_t\rangle / 2\Delta t - \Lambda/\Delta t`
    (symmetrised; fitted rows only).  Returns ``(Lambda, D)``, ``(d, d)`` each.
    """
    import jax.numpy as jnp
    from SFI.trajectory import TimeSeriesExtra

    lag1, sq, n1, n0, dts = 0.0, 0.0, 0, 0, []
    for ds in coll.datasets:
        X = np.asarray(ds.X)
        dt = float(np.mean(np.asarray(ds.dt)))
        rows = np.asarray(ds.dynamic_mask if ds.dynamic_mask is not None else ds.mask, bool)
        valid = np.asarray(ds.mask, bool)
        F = []
        for t in range(X.shape[0] - 1):
            extras = {k: jnp.asarray(v.data[t] if isinstance(v, TimeSeriesExtra) else v)
                      for k, v in (ds.extras_global or {}).items()}
            F.append(np.asarray(inf.force_inferred(jnp.asarray(np.nan_to_num(X[t])), extras=extras)))
        r = X[1:] - X[:-1] - np.asarray(F) * dt
        ok = rows[:-1] & valid[1:] & valid[:-1]
        r = np.where(ok[..., None], r, 0.0)
        both = ok[1:] & ok[:-1]
        lag1 = lag1 + np.einsum("tnm,tnk->mk", np.where(both[..., None], r[1:], 0.0), r[:-1])
        sq = sq + np.einsum("tnm,tnk->mk", r, r)
        n1, n0 = n1 + both.sum(), n0 + ok.sum()
        dts.append(dt)
    dt = float(np.mean(dts))
    L = -lag1 / n1
    L = 0.5 * (L + L.T)
    return L, sq / n0 / (2 * dt) - L / dt

def periodic_tracks_collection(X, dt, box, *, cutoff, lag=2):
    """Simulated tracks in a periodic box, with per-frame Verlet lists."""
    from SFI import TrajectoryCollection
    from SFI.trajectory import TimeSeriesExtra
    from SFI.utils.neighbors import build_neighbor_csr, pad_neighbor_csr

    d = X[lag:, :, :2] - X[:-lag, :, :2]
    skin = 2.0 * np.hypot(*np.moveaxis(d - box * np.round(d / box), -1, 0)).max()
    lists = [build_neighbor_csr(np.mod(f[:, :2], box), cutoff + skin, box) for f in X]
    nnz = max(len(ix) for _, ix in lists)
    extras = {"box": np.asarray(box), "indptr": TimeSeriesExtra(np.stack([ip for ip, _ in lists])),
              "indices": TimeSeriesExtra(np.stack([pad_neighbor_csr(ip, ix, nnz)[1] for ip, ix in lists]))}
    return TrajectoryCollection.from_arrays(X=X, dt=dt, extras_global=extras)

def sample_pair_configurations(X, box, cutoff, n, rng):
    """``n`` random ``(r, θ1, θ2)`` of pairs closer than ``cutoff`` in periodic tracks."""
    out = []
    for f in X:
        p = cKDTree(np.mod(f[:, :2], box), boxsize=box).query_pairs(cutoff, output_type="ndarray")
        p = np.concatenate([p, p[:, ::-1]])
        d = f[p[:, 0], :2] - f[p[:, 1], :2]
        d -= box * np.round(d / box)
        phi_r = np.arctan2(d[:, 1], d[:, 0])
        out.append(np.stack([np.hypot(d[:, 0], d[:, 1]), f[p[:, 0], 2] - phi_r, f[p[:, 1], 2] - phi_r], 1))
    out = np.concatenate(out)
    return out[rng.choice(len(out), n, replace=False)]

__all__ = [
    "DATA_URL", "load_set_a", "tracks_to_arrays", "per_frame_neighbours", "interior_rows",
    "tracks_collection", "tapered_polyexp_kernels", "angular_modes", "orientation_features",
    "interaction_reader", "fitted_pair_force", "simulate_fitted", "residual_noise", "periodic_tracks_collection", "sample_pair_configurations",
    "polar_order_vs_area", "polarity_autocorrelation", "number_fluctuations",
    "body_frame_density", "relative_orientations",
]

## The experiment

Silica spheres of diameter $\sigma = 3.17\,\mu$\ m, half-coated
with titanium, swim metal side first under a vertical AC field and
organise into a polar flock.  The movie of this experiment published
with Iwasawa *et al.* (arXiv:2011.14548, ancillary file, 30 fps):

.. raw:: html

   <video controls loop muted playsinline width="100%"
          src="https://arxiv.org/src/2011.14548v1/anc/3_Ordered.mp4"></video>

The tracks — position, heading $\varphi$ and a track label per
detected particle and frame — come from the authors' repository (5 MB,
read without executing code from the file).  Particles enter and leave
the field of view, so the tracks become a masked collection, with
per-frame neighbour lists and a fit mask that keeps particles near the
edges (whose neighbours were partly not recorded) as neighbours only:

In [ ]:
frames, dt, fov = load_set_a(cache_dir)
R_C = 21.0                                          # interaction range (µm), as in the paper
coll = tracks_collection(frames, dt, fov, cutoff=R_C, margin=40.0)
print(f"{len(frames)} frames at dt = {dt:.4f} s, {coll.datasets[0].X.shape[1]} tracks, "
      f"{np.mean([len(f) for f in frames]):.0f} particles per frame")

X_obs, present = tracks_to_arrays(frames)
fig_tr, ax_tr = dark_fig(figsize=(6.5, 5.2))
anim_tracks = animate_particles(TrajectoryCollection.from_arrays(X=X_obs, mask=present, dt=dt),
                                color_dim=2, cmap="hsv", vmin=0, vmax=2 * np.pi, s=4.0,
                                interval=1000 * dt, ax=ax_tr)
_particle_axes(ax_tr, fov)
ax_tr.set_title("tracks, coloured by heading", color="#D0D0D0")
plt.show()

## The generic interaction basis

The effect of particle 2 on particle 1 is a radial velocity
$v_r$, a transverse velocity $v_\theta$ and a torque
$\omega$, functions of the distance $r$ and of the headings
$\theta_1, \theta_2$ measured from $\mathbf r_1 - \mathbf
r_2$.  The paper expands them as
$\sum_l g_l(r) \sum_{|k_1|+|k_2| \le N_a} a_{l,k_1k_2}
\cos\text{ or }\sin(k_1\theta_1 + k_2\theta_2)$ — cosines for
$v_r$, sines for $v_\theta$ and $\omega$, as parity
requires — with $g_l(r) = r^l e^{-r}/l!$ ($r$ in µm,
$l = 0, 2, \ldots, 14$, as in the paper's code).  In SFI this is
the product of a radial and an angular pair interactor; we multiply the
radial functions by a window that vanishes smoothly at $r_c$ and
below $1.5\,\sigma$, where the tracks hold almost no pairs.
Self-propulsion and three more one-body terms complete the basis.

In [ ]:
from SFI.bases import unit_axes, x_components
from SFI.bases.pairs import heading_vector, scalar_pair_basis
from SFI.statefunc import Basis

kernels = tapered_polyexp_kernels(cutoff=R_C, onset=(1.0 * SIGMA, 1.5 * SIGMA))

def pair_interactions(Na, box=None):
    return scalar_pair_basis(kernels, dim=3, box=box, spatial_dims=slice(0, 2)) * orientation_features(Na, box=box)

def pair_basis(Na, box=None):
    return pair_interactions(Na, box).dispatch_pairs_from_extras(
        indptr_key="indptr", indices_key="indices", return_as="basis")

x, y, phi = x_components(3)
e_x, e_y, _ = unit_axes(3)
e_heading = heading_vector(dim=3, angle_index=2)                                   # U ê_φ
e_side = phi.elementwisemap(jnp.sin) * e_x - phi.elementwisemap(jnp.cos) * e_y     # ê_φ⊥
one_body = [e_x, e_y, e_heading, e_side]

pairs_2 = pair_basis(Na=2)
basis = Basis.stack([*one_body, pairs_2])
print(f"{basis.n_features} features, {pairs_2.n_features} of them pair interactions")

## Inference

Linear SFI, on every frame.  The tracks carry localisation noise (we
measure it below), so we use **Itô-shift** moments — the basis is
evaluated one frame before each displacement, whose noise it then does
not see — with the trapeze Gram matrix.  The ill-conditioned radial
functions (SFI warns about the Gram matrix) are why this script runs JAX
in double precision.

In [ ]:
from SFI import OverdampedLangevinInference

inf = OverdampedLangevinInference(coll)
inf.compute_diffusion_constant(method="WeakNoise")
inf.infer_force_linear(basis, M_mode="Ito-shift", G_mode="trapeze")
inf.compute_force_error()

U, U_cov = inf.coeff_block(e_heading)
U = float(np.ravel(U)[0])
print(f"self-propulsion U = {U:.2f} ± {float(np.sqrt(np.ravel(U_cov)[0])):.2f} µm/s")

## The learned interactions

The analogue of the paper's Fig. 2, at $r = 2.5\,\sigma$, the most
frequent neighbour distance.  Left: how often each pair of relative
orientations occurs — neighbours mostly move in the same direction
($\theta_1 \approx \theta_2$).  Hatching marks where the 95 %
confidence interval is wider than the value.

In [ ]:
read = interaction_reader(pair_interactions(2))
value, stderr = read(*inf.coeff_block(pairs_2), 2.5 * SIGMA, T1, T2)
t1_obs, t2_obs = relative_orientations(frames, 2.25 * SIGMA, 2.75 * SIGMA)

scales = [1 / U, 1 / U, SIGMA / U]
fig_maps, axes_maps = plt.subplots(1, 4, figsize=(16, 3.9))
h = axes_maps[0].hist2d(t1_obs, t2_obs, bins=36, range=[[0, 2 * np.pi]] * 2, cmap="magma", rasterized=True)
plt.colorbar(h[3], ax=axes_maps[0], label="pairs")
axes_maps[0].set_title(r"pairs at $r \approx 2.5\,\sigma$", fontsize=11)
_angle_axes(axes_maps[0])
for j, ax in enumerate(axes_maps[1:]):
    im = ax.pcolormesh(TH, TH, (value[..., j] * scales[j]).T, cmap="RdBu_r", vmin=-0.3, vmax=0.3,
                       shading="auto", rasterized=True)
    with plt.rc_context({"hatch.color": "#B0B0B0"}):
        ax.contourf(TH, TH, (1.96 * stderr[..., j] > np.abs(value[..., j])).T.astype(float),
                    levels=[0.5, 1.5], hatches=["xxx"], colors="none")
    _angle_axes(ax)
    ax.set_title(TITLES[j], fontsize=11)
fig_maps.colorbar(im, ax=axes_maps[1:], shrink=0.85)
plt.show()

A repulsion strongest when particle 1 is in front of particle 2 and
turned sideways, a weak transverse velocity, and a torque that turns
particle 1 away from particle 2, most strongly when particle 2 faces it
($\theta_2 \approx 0$) — the "modulated turning away" of the
paper.  Along $r$, with the distribution of pair distances:

In [ ]:
r = np.linspace(1.3 * SIGMA, 6.5 * SIGMA, 120)
profiles = {t1: read(*inf.coeff_block(pairs_2), r, t1, 0.0) for t1 in (np.pi / 2, 3 * np.pi / 2)}

from scipy.spatial import cKDTree

r_pairs = np.concatenate([np.hypot(*(f[p[:, 0], :2] - f[p[:, 1], :2]).T) for f in frames[::10]
                          for p in [cKDTree(f[:, :2]).query_pairs(6.5 * SIGMA, output_type="ndarray")]])
fig_rad, axes_rad = plt.subplots(1, 2, figsize=(12, 4))
for ax, j in [(axes_rad[0], 0), (axes_rad[1], 2)]:
    ax_h = ax.twinx()
    ax_h.hist(r_pairs / SIGMA, bins=80, range=(1.3, 6.5), color="#808080", alpha=0.15)
    ax_h.set_yticks([]); ax_h.spines["right"].set_visible(False)
    ax.set_zorder(ax_h.get_zorder() + 1); ax.patch.set_visible(False)
    for (t1, (v, e)), col, lab in zip(profiles.items(), [SFI_COLORS["inferred"], SFI_COLORS["highlight"]],
                                      [r"$(\theta_1, \theta_2) = (\pi/2, 0)$", r"$(3\pi/2, 0)$"]):
        ax.plot(r / SIGMA, v[:, j] * scales[j], color=col, lw=2, label=lab)
        ax.fill_between(r / SIGMA, (v[:, j] - 1.96 * e[:, j]) * scales[j], (v[:, j] + 1.96 * e[:, j]) * scales[j],
                        color=col, alpha=0.25)
    ax.axhline(0, color="#808080", lw=0.8)
    ax.set_xlabel(r"$r/\sigma$"); ax.set_ylabel(TITLES[j]); ax.legend(fontsize=9)
plt.show()

## How much noise, and does it matter?

SFI's measurement-noise estimate $\hat\Lambda = -\langle \Delta
X_t\,\Delta X_{t-1}\rangle$ comes out *negative* on the positions:
successive displacements of a swimmer are correlated, and that
persistence, $\langle F_t F_{t-1}\rangle\Delta t^2$, hides the
noise.  On the residuals of the fit, $r_t = \Delta X_t - \hat
F(X_t)\Delta t$, the drift is gone and the same estimator works:

In [ ]:
Lam, D_res = residual_noise(inf, coll)
noise_sd = np.sqrt(np.clip(np.diag(Lam), 0, None))
D_r = D_res[2, 2]
print("Λ̂ from displacements (x, y, φ):", np.diag(np.asarray(inf.Lambda)).round(4))
print("Λ̂ from residuals     (x, y, φ):", np.diag(Lam).round(4))
print(f"localisation noise {noise_sd[0]:.2f}, {noise_sd[1]:.2f} µm and {np.degrees(noise_sd[2]):.1f}°;  "
      f"rotational diffusion D_r = {D_r:.3f} 1/s")

About two pixels of positional noise — as large as the distance a
colloid swims in a frame — and a few degrees on the headings.  To see
what that does to the inference, we let the learned model generate data
with a known answer: simulate it from the first frame (periodic box of
the size of the field of view, the rotational diffusion just measured,
translational diffusion neglected as in the paper), add noise of the
measured size, and
refit — with Itô-shift, and with the paper's convention (Stratonovich
moments, rectangle Gram matrix, ``preset="legacy-sfi-v1.0"``).

In [ ]:
D_sim = np.diag([0.0, 0.0, D_r])
x0 = frames[0][:, :3]
sim = dict(box=fov, dt=dt, cutoff=R_C)
X_true = simulate_fitted(inf, one_body, pairs_2, kernels, 2, x0, 10 + len(frames) * dt, D=D_sim,
                         key=jax.random.PRNGKey(1), **sim)[-len(frames):]
rng = np.random.default_rng(0)
X_noisy = X_true + noise_sd * rng.standard_normal(X_true.shape)

configs = sample_pair_configurations(X_true[::10], fov, R_C, 20000, rng)
v_true, _ = read(inf.coeff_block(pairs_2)[0], None, *configs.T)
print(f"{'refit on simulated noisy tracks':34s}    U    error on  v_r   v_θ    ω   (true U = {U:.2f})")
for label, opts in [("Itô-shift, trapeze", dict(M_mode="Ito-shift", G_mode="trapeze")),
                    ("paper: Stratonovich, rectangle", dict(preset="legacy-sfi-v1.0"))]:
    pairs_b = pair_basis(2, box="extras")
    refit = OverdampedLangevinInference(periodic_tracks_collection(X_noisy, dt, fov, cutoff=R_C))
    refit.compute_diffusion_constant(method="WeakNoise")
    refit.infer_force_linear(Basis.stack([*one_body, pairs_b]), **opts)
    v_fit, _ = read(refit.coeff_block(pairs_b)[0], None, *configs.T)
    err = np.sqrt(np.mean((v_fit - v_true) ** 2, 0) / np.mean(v_true ** 2, 0))
    print(f"{label:34s} {float(np.ravel(refit.coeff_block(e_heading)[0])[0]):6.2f}"
          f"   {'':9s}{err[0]:5.2f} {err[1]:5.2f} {err[2]:5.2f}")

(Relative RMS errors over the simulated pairs.)  Itô-shift recovers the
self-propulsion and, to within roughly 15–30 %, the interactions; the
transverse velocity, the weakest of the three, is the least precise.  The
Stratonovich moments of the paper are thrown off by the noise on the
positions: their Itô-to-Stratonovich correction rests on a per-step
diffusion estimate that, for particles swimming at $U$, is
dominated by $U^2\Delta t$.

## Does the learned model flock?

As in the paper we fit models of angular order $N_a = 0$
(isotropic radial interactions only), 1 and 2, and simulate each for
60 s from the first experimental frame.  The flock is a balance between
turning-away torques and rotational noise, so we also run the
first-order model with the paper's rotational diffusion,
$D_r = 0.092$ s⁻¹.

In [ ]:
models = {2: (inf, pairs_2)}
for Na in (0, 1):
    pairs_Na = pair_basis(Na)
    models[Na] = (OverdampedLangevinInference(coll), pairs_Na)
    models[Na][0].compute_diffusion_constant(method="WeakNoise")
    models[Na][0].infer_force_linear(Basis.stack([*one_body, pairs_Na]), M_mode="Ito-shift", G_mode="trapeze")

def run(Na, D, seed):
    fitted, pairs_Na = models[Na]
    return simulate_fitted(fitted, one_body, pairs_Na, kernels, Na, x0, 60.0, D=D, key=jax.random.PRNGKey(seed), **sim)

runs = {Na: run(Na, D_sim, 10 + Na) for Na in (0, 1, 2)}
runs["1, paper D_r"] = run(1, np.diag([0.0, 0.0, 0.092]), 11)
for name, X in runs.items():
    print(f"N_a = {name}: polar order after 60 s = {np.abs(np.exp(1j * X[-1, :, 2]).mean()):.2f}")

The experiment (left) and the second-order model (right), from the same
frame, over the 6.7 s of the recording:

In [ ]:
wrapped = runs[2][:len(frames)].copy()
wrapped[..., :2] = np.mod(wrapped[..., :2], fov)
fig_mov, axes_mov = dark_fig(1, 2, figsize=(9, 4))
anim_cmp = animate_particles_comparison(
    [TrajectoryCollection.from_arrays(X=X_obs, mask=present, dt=dt),
     TrajectoryCollection.from_arrays(X=wrapped, dt=dt)],
    titles=["experiment", r"learned model ($N_a = 2$)"], color_dim=2, cmap="hsv",
    vmin=0, vmax=2 * np.pi, s=3.0, interval=1000 * dt, axes=axes_mov)
_particle_axes(axes_mov, fov)
plt.show()

Statistics the fit never used: the polar order $C(S)$ in square
regions of area $S$ (second half of each run), the polarity
autocorrelation $f_p(\tau) = \langle\cos[\varphi(t+\tau) -
\varphi(t)]\rangle$, and the density of neighbours around a particle in
its own frame.

In [ ]:
sides = np.array([10.0, 20.0, 40.0, 84.0, 168.0])
lags = np.arange(0, 451, 5)
C_data = polar_order_vs_area(frames, fov, sides)
fp_data = polarity_autocorrelation(np.where(present, X_obs[..., 2], np.nan), lags[lags < len(frames)])
C_sim = {Na: polar_order_vs_area(list(X[len(X) // 2:]), fov, sides) for Na, X in runs.items()}
fp_sim = {Na: polarity_autocorrelation(X[..., 2], lags[lags < len(X)]) for Na, X in runs.items()}
rho_data = body_frame_density(frames, fov)
rho_sim = body_frame_density(list(runs[2][len(runs[2]) // 2::5]), fov, periodic=True)

colors = {0: SFI_COLORS["tertiary"], 1: SFI_COLORS["highlight"], 2: SFI_COLORS["inferred"],
          "1, paper D_r": SFI_COLORS["error"]}
fig_obs, (ax_C, ax_f) = plt.subplots(1, 2, figsize=(12, 4.2))
ax_C.loglog(sides ** 2, C_data, "o-", color=SFI_COLORS["data"], lw=2, label="experiment")
ax_f.plot(lags[lags < len(frames)] * dt, fp_data, color=SFI_COLORS["data"], lw=3, label="experiment")
for Na in runs:
    lab = rf"$N_a = 1$, $D_r = 0.092$" if Na == "1, paper D_r" else f"model $N_a = {Na}$"
    ax_C.loglog(sides ** 2, C_sim[Na], "s--", color=colors[Na], label=lab)
    ax_f.plot(lags[:len(fp_sim[Na])] * dt, fp_sim[Na], "--", color=colors[Na], label=lab)
ax_C.loglog(sides ** 2, C_sim[0][0] * sides[0] / sides, ":", color="#808080", lw=1, label=r"$\propto S^{-1/2}$")
ax_C.set_xlabel(r"$S$ [$\mu$m$^2$]"); ax_C.set_ylabel(r"$C(S)$"); ax_C.set_title("polar order")
ax_f.set_xlabel(r"$\tau$ [s]"); ax_f.set_ylabel(r"$f_p(\tau)$"); ax_f.set_title("polarity autocorrelation")
ax_f.axhline(0, color="#808080", lw=0.8)
for ax in (ax_C, ax_f):
    ax.legend(fontsize=9)
plt.show()

fig_rho, axes_rho = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, rho, lab in [(axes_rho[0], rho_data, "experiment"), (axes_rho[1], rho_sim, r"model $N_a = 2$")]:
    im = ax.imshow(rho.T, origin="lower", extent=[-25 / SIGMA, 25 / SIGMA] * 2, cmap="magma", vmin=0, vmax=1.6)
    ax.add_patch(plt.Circle((0, 0), 0.5, color=SFI_COLORS["inferred"]))
    ax.set_xlabel(r"$x_\parallel/\sigma$ (along the heading)"); ax.set_ylabel(r"$x_\perp/\sigma$"); ax.set_title(lab)
fig_rho.colorbar(im, ax=axes_rho, shrink=0.85, label="density / mean density")
plt.show()

Isotropic interactions alone ($N_a = 0$) do not flock: the order
of the first frame is lost and $C(S)$ falls as $S^{-1/2}$.
The second-order model reproduces the experiment — the polar order at
all region sizes, the polarity autocorrelation over the recording, the
ring of neighbours at $r \approx 2.5\,\sigma$.  The first-order
model keeps a weaker order with the rotational diffusion measured here
and none with the paper's larger $D_r$: as in the paper, the
second angular order is needed.

## Comparison with the paper

With 3 % of the data the qualitative picture of the paper is recovered —
the self-propulsion speed, a mostly isotropic repulsion, a weak transverse
velocity, a modulated turning-away torque, and a flocking simulation.
Where we differ:

* **Localisation noise.**  About two pixels on positions, not "below one
  pixel".  At this level the paper's Stratonovich estimator overestimates
  $U$ and misses the radial velocity substantially in the
  bootstrap above; the amplitudes of its Fig. 2 are uncertain to that
  extent.
* **Rotational diffusion.**  Measured on the residuals of the fit,
  $D_r$ comes out about a third below the paper's 0.092 s⁻¹.  The
  conclusion that the second angular order is needed holds at both values,
  but how much order the first-order model keeps depends on it.
* **Radial basis.**  The paper's text describes $g_l$ with
  $l = 0 \ldots 7$ and $r_0 = 2\,\mu$\ m; its code — and this
  page — use $l = 0, 2, \ldots, 14$ with $r_0 = 1\,\mu$\ m.
  The code also truncates them at $r_c$ while still nonzero, which
  biases Stratonovich moments; the window here vanishes smoothly.

## Thumbnail

In [ ]:
fig_thumb, ax_t = plt.subplots(figsize=(4, 4))
ax_t.pcolormesh(TH, TH, (value[..., 2] * scales[2]).T, cmap="RdBu_r", vmin=-0.25, vmax=0.25,
                shading="auto", rasterized=True)
ax_t.set_xticks([]); ax_t.set_yticks([])
for sp in ax_t.spines.values():
    sp.set_visible(False)
plt.show()